## Tratamento dos dados

Nesse notebook os dados são tratados de forma a tornar possível a análise

In [1]:
import pandas as pd
from utils.load_csv import load_csv
from utils.load_shp import load_shp
from utils.save_csv import save_csv
from utils.save_shp import save_shp

In [2]:
df_tse_2026 = load_csv('resultados_2026.csv')

df_observasampa = load_csv('observasampa.csv')

gdf_distritos = load_shp('distritos.shp')

In [3]:
# Resultado do segundo turno da eleição presidencial brasileira por seção eleitoral do país
# Seleciona as variáveis de interesse e filtra apenas o segundo turno na cidade de São Paulo

variaveis = ['NR_TURNO',
             'SG_UF',
             'CD_MUNICIPIO',
             'NM_MUNICIPIO',
             'NR_ZONA',
             'NR_SECAO',
             'CD_CARGO',
             'DS_CARGO',
             'NR_VOTAVEL',
             'NM_VOTAVEL',
             'QT_VOTOS',
             'NR_LOCAL_VOTACAO',
             'NM_LOCAL_VOTACAO',
             'DS_LOCAL_VOTACAO_ENDERECO']

df_tse_2026_filtrado = (df_tse_2026.loc[
    (df_tse_2026['NR_TURNO'] == 2) &
    (df_tse_2026['NM_MUNICIPIO'] == 'SÃO PAULO'),
    variaveis].copy())

In [4]:
# Indicadores socioeconômicos do município de São Paulo por distrito
# Seleciona as variáveis de interesse, filtra apenas os distritos e o período de 2024

variaveis = ["01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família",
             "População total"]

df_observasampa_filtrado = df_observasampa[df_observasampa["Nome"].isin(variaveis)
                                           & df_observasampa["Região"].str.endswith("(Distrito)")
                                           & (df_observasampa["Período"] == 2024)].copy()

df_observasampa_filtrado["Resultado"] = pd.to_numeric(df_observasampa_filtrado["Resultado"])

df_observasampa_filtrado = (df_observasampa_filtrado
                            .pivot(index="Região", columns="Nome", values="Resultado")
                            .rename(columns={"01.03.02 Quantidade de famílias beneficiárias do Programa Bolsa Família": "familias_pbf",
                                             "População total": "pop_total"})
                            .rename_axis(columns=None)     
                            .reset_index()
                            .rename(columns={"Região": "distrito"}))

df_observasampa_filtrado["distrito"] = df_observasampa_filtrado["distrito"].str.replace(r"\s*\(Distrito\)$", "", regex=True)

In [5]:
# Malha territorial dos limites administrativos dos distritos do estado de São Paulo
# Seleciona as variáveis de interesse e filtra apenas os distritos da cidade de São Paulo

gdf_distritos_filtrado = gdf_distritos.loc[gdf_distritos["CD_MUN"] == "3550308",
                                           ["CD_DIST", "NM_DIST", "geometry"]].reset_index(drop=True)

In [6]:
save_csv(df_tse_2026_filtrado, 'resultados_2026_filtrado.csv')
save_csv(df_observasampa_filtrado, 'observasampa_filtrado.csv')
save_shp(gdf_distritos_filtrado, 'distritos_filtrado.shp')

Base salva em data\resultados_2026_filtrado.csv
Base salva em data\observasampa_filtrado.csv
Shapefile salvo em data\distritos_filtrado.shp
